# COMP 9130 — Capstone Project: Military Camouflage Object Detection
## Notebook 03: Transfer Learning — Experiment 2 (SegFormer-B2)

**Author:** Sepehr Mansouri (Experiment 2 + Evaluation Lead)   
**Datasets:** COD10K (pretrain) → ACD1K (fine-tune)   
**Model:** SegFormer-B2 (nvidia/segformer-b2-finetuned-ade-512-512)

**Purpose:** Train SegFormer-B2 via two-stage cross-domain transfer learning.
Stage 1 pretrains on COD10K (animal camouflage) to acquire general camouflage
representations, then Stage 2 fine-tunes on ACD1K (military camouflage) with
a lower learning rate to adapt to the target domain while minimising
catastrophic forgetting. This tests whether animal-camouflage pretraining
transfers to military-specific targets.

---

### Notebook Structure

1. **Environment Setup** — Clone repo, install dependencies, configure Kaggle credentials
2. **Dataset Download** — COD10K (Kaggle), ACD1K (Kaggle)
3. **Dataset Verification** — Confirm all folder paths and split index files
4. **GPU Verification** — Confirm A100 is available
5. **Stage 1 — COD10K Pretraining** — Train on animal camouflage with differential LR
6. **Stage 1 Results** — Inspect best checkpoint
7. **Stage 2 — Hyperparameter Sweep** — Three 20-epoch runs on ACD1K across learning rates [1e-4, 1e-5, 5e-6]
8. **Sweep Results** — Compare val mIoU across runs, select best LR
9. **Stage 2 — Final Training Run** — 50-epoch run with best LR, early stopping patience=10
10. **Results & Export** — Save history.json, push to GitHub, download checkpoint

---

### Experiment Design

| Setting | Value |
|---|---|
| Model | SegFormer-B2 |
| Pretrained checkpoint | nvidia/segformer-b2-finetuned-ade-512-512 |
| **Stage 1 — COD10K Pretrain** | |
| Training data | COD10K (5,950 images) |
| Validation data | COD10K val (3,950 images) |
| Encoder LR | 1e-4 |
| Decode head LR | 1e-3 |
| **Stage 2 — ACD1K Fine-tune** | |
| Training data | ACD1K (748 images) |
| Validation data | ACD1K val (230 images) |
| Sweep LRs | 1e-4, 1e-5, 5e-6 |
| Final LR | TBD (best from sweep) |
| **Common** | |
| Input resolution | 512 × 512 |
| Batch size | 16 (A100) |
| Loss function | Binary cross-entropy + Dice |
| Early stopping patience | 10 epochs |
| Hardware | Google Colab A100 GPU |

---
## 1. Environment Setup

In [ ]:
# Clone repo
!git clone https://github.com/bing-er/AI-final-project.git
%cd /content/AI-final-project

fatal: destination path 'AI-final-project' already exists and is not an empty directory.
/content/AI-final-project


In [ ]:
# Install dependencies
!pip install -q transformers albumentations kaggle

In [ ]:
# Must Upload kaggle.json (run once per session)
import os
from google.colab import files

files.upload()  # select kaggle.json from your machine when prompted
os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('Kaggle credentials configured')

Saving kaggle.json to kaggle.json
Kaggle credentials configured


---
## 2. Dataset Download

Experiment 2 uses only **COD10K** and **ACD1K** (no CAMO needed).

In [ ]:
# Download COD10K (~1.2 GB)
import os
if not os.path.exists('data/COD10K-v3/Train/Image'):
    !kaggle datasets download \
        -d ivanomelchenkoim11/cod10k-dataset \
        -p data/ --unzip
else:
    print("✅ COD10K already exists, skipping download")

✅ COD10K already exists, skipping download


In [ ]:
# Download ACD1K (~350 MB)
import os
if not os.path.exists('data/dataset-splitM/Training/images'):
    !kaggle datasets download \
        -d aalihhiader/military-camouflage-soldiers-dataset-mcs1k \
        -p data/ --unzip
else:
    print("✅ ACD1K already exists, skipping download")

✅ ACD1K already exists, skipping download


---
## 3. Dataset & Split Verification

In [ ]:
# Verify folder structure
import os
expected = [
    'data/COD10K-v3/Train/Image',
    'data/COD10K-v3/Train/GT_Object',
    'data/COD10K-v3/Test/Image',
    'data/COD10K-v3/Test/GT_Object',
    'data/dataset-splitM/Training/images',
    'data/dataset-splitM/Training/GT',
    'data/dataset-splitM/Testing/images',
    'data/dataset-splitM/Testing/GT',
]
for p in expected:
    status = '✅' if os.path.exists(p) else '❌ NOT FOUND'
    print(f'{status} — {p}')

✅ — data/COD10K-v3/Train/Image
✅ — data/COD10K-v3/Train/GT_Object
✅ — data/COD10K-v3/Test/Image
✅ — data/COD10K-v3/Test/GT_Object
✅ — data/dataset-splitM/Training/images
✅ — data/dataset-splitM/Training/GT
✅ — data/dataset-splitM/Testing/images
✅ — data/dataset-splitM/Testing/GT


---
## 4. GPU Verification

In [ ]:
# Verify GPU
!nvidia-smi

Sun Apr  5 00:45:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# Verify dataset loading (COD10K and ACD1K conditions)
!PYTHONPATH=/content/AI-final-project/src \
 python /content/AI-final-project/src/dataset.py \
    /content/AI-final-project/data/ \
    /content/AI-final-project/splits/

Verifying all conditions and splits...

--- ACD1K / train ---
  [ACD1K] 748 images (folder-scan mode)
  [DataLoader] condition=acd1k split=train samples=748 batches=187
  image : torch.Size([4, 3, 512, 512])  mask  : torch.Size([4, 1, 512, 512])  values: [0.0, 1.0]  datasets: {'ACD1K'}
  ✅ OK

--- ACD1K / val ---
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] condition=acd1k split=val samples=230 batches=58
  image : torch.Size([4, 3, 512, 512])  mask  : torch.Size([4, 1, 512, 512])  values: [0.0, 1.0]  datasets: {'ACD1K'}
  ✅ OK

--- COD10K / train ---
  [Splits] COD10K train: 5950 images (from cod10k_splits.json)
  [COD10K] 5950 images (split-file mode)
  [DataLoader] condition=cod10k split=train samples=5950 batches=1487
  image : torch.Size([4, 3, 512, 512])  mask  : torch.Size([4, 1, 512, 512])  values: [0.0, 1.0]  datasets: {'COD10K'}
  ✅ OK

--- COD10K / val ---
  [Splits] COD10K val: 3950 images (from cod10k_split

---
## 5. Stage 1 — COD10K Pretraining

Stage 1 pretrains SegFormer-B2 on COD10K (5,950 animal camouflage images)
to learn general camouflage-detection features before domain-specific
fine-tuning on military imagery.

**Differential learning rate:** The MiT-B2 encoder (initialised from ADE20K)
uses a lower LR (1e-4) to preserve pretrained spatial features, while the
randomly-initialised binary decode head uses a higher LR (1e-3) to train
faster from scratch.

First, we need to add `train_exp2.py` to the repo if it's not already there.

In [ ]:
# Check if train_exp2.py exists, upload if not
import os
if not os.path.exists('src/train_exp2.py'):
    print("train_exp2.py not found in src/ — uploading...")
    from google.colab import files
    uploaded = files.upload()  # upload train_exp2.py
    os.rename('train_exp2.py', 'src/train_exp2.py')
    print("✅ train_exp2.py placed in src/")
else:
    print("✅ train_exp2.py already exists in src/")

✅ train_exp2.py already exists in src/


In [ ]:
# Stage 1: COD10K pretraining (differential LR)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp2.py \
    --stage 1 \
    --encoder_lr 1e-4 --head_lr 1e-3 \
    --epochs 50 --batch_size 16 --accum_steps 1 \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp2/stage1

Device: cuda
Config saved → outputs/exp2/stage1/config.json
{
  "stage": 1,
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp2/stage1",
  "encoder_lr": 0.0001,
  "head_lr": 0.001,
  "lr": 1e-05,
  "stage1_weights": null,
  "weight_decay": 0.0001,
  "epochs": 50,
  "batch_size": 16,
  "accum_steps": 1,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [Splits] COD10K train: 5950 images (from cod10k_splits.json)
  [COD10K] 5950 images (split-file mode)
  [DataLoader] condition=cod10k split=train samples=5950 batches=371
  [Splits] COD10K val: 3950 images (from cod10k_splits.json)
  [COD10K] 3950 images (split-file mode)
  [DataLoader] condition=cod10k split=val samples=3950 batches=247

[Model] Loading SegFormer-B2 (ADE20K pretrained)...
config.json: 6.88kB [00:00, 35.6MB/s]
pytorch_model.bin: 100% 110M/110M [00:02<00:00, 54.5MB/s]
Loading weights:  90% 342/380 [00:00<00:00, 982.10it/s, Materializing param=segformer.encoder.block.3.2.atte

---
## 6. Stage 1 Results

In [ ]:
# Stage 1 results
import json

with open('outputs/exp2/stage1/history.json') as f:
    history = json.load(f)

best = max(history, key=lambda x: x['val_mIoU'])
print(f"Stage 1 (COD10K pretrain):")
print(f"  Best epoch : {best['epoch']}")
print(f"  val mIoU   : {best['val_mIoU']:.4f}")
print(f"  val F1     : {best['val_F1']:.4f}")
print(f"  val MAE    : {best['val_MAE']:.4f}")

Stage 1 (COD10K pretrain):
  Best epoch : 43
  val mIoU   : 0.8944
  val F1     : 0.8464
  val MAE    : 0.0162


---
## 7. Stage 2 — ACD1K Fine-Tuning (Hyperparameter Sweep)

Stage 2 loads the best Stage-1 checkpoint and fine-tunes on ACD1K (748
military camouflage images) with a uniformly lower learning rate to adapt
to the military domain while minimising catastrophic forgetting of the
COD10K camouflage features.

We sweep over three learning rates: **1e-4, 1e-5, 5e-6** with 20 epochs
each to find the best LR before the final 50-epoch run.

In [ ]:
# Stage 2 sweep run 1 (lr=1e-4)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp2.py \
    --stage 2 \
    --lr 1e-4 --epochs 20 --batch_size 16 --accum_steps 1 \
    --stage1_weights outputs/exp2/stage1/best_model.pth \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp2/sweep_s2_lr1e4

Device: cuda
Config saved → outputs/exp2/sweep_s2_lr1e4/config.json
{
  "stage": 2,
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp2/sweep_s2_lr1e4",
  "encoder_lr": 0.0001,
  "head_lr": 0.001,
  "lr": 0.0001,
  "stage1_weights": "outputs/exp2/stage1/best_model.pth",
  "weight_decay": 0.0001,
  "epochs": 20,
  "batch_size": 16,
  "accum_steps": 1,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [ACD1K] 748 images (folder-scan mode)
  [DataLoader] condition=acd1k split=train samples=748 batches=46
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] condition=acd1k split=val samples=230 batches=15

[Model] Loading SegFormer-B2 (ADE20K pretrained)...
Loading weights: 100% 380/380 [00:00<00:00, 851.73it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]
SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key        

In [41]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [42]:
import json

with open('outputs/exp2/sweep_s2_lr1e4/history.json') as f:
    history = json.load(f)

best = max(history, key=lambda x: x['val_mIoU'])
print(f"Stage 2 sweep (lr=1e-4):")
print(f"  Best epoch : {best['epoch']}")
print(f"  val mIoU   : {best['val_mIoU']:.4f}")
print(f"  val F1     : {best['val_F1']:.4f}")
print(f"  val MAE    : {best['val_MAE']:.4f}")

Stage 2 sweep (lr=1e-4):
  Best epoch : 12
  val mIoU   : 0.8682
  val F1     : 0.8622
  val MAE    : 0.0427


In [ ]:
# Stage 2 sweep run 2 (lr=1e-5)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp2.py \
    --stage 2 \
    --lr 1e-5 --epochs 20 --batch_size 16 --accum_steps 1 \
    --stage1_weights outputs/exp2/stage1/best_model.pth \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp2/sweep_s2_lr1e5

Device: cuda
Config saved → outputs/exp2/sweep_s2_lr1e5/config.json
{
  "stage": 2,
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp2/sweep_s2_lr1e5",
  "encoder_lr": 0.0001,
  "head_lr": 0.001,
  "lr": 1e-05,
  "stage1_weights": "outputs/exp2/stage1/best_model.pth",
  "weight_decay": 0.0001,
  "epochs": 20,
  "batch_size": 16,
  "accum_steps": 1,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [ACD1K] 748 images (folder-scan mode)
  [DataLoader] condition=acd1k split=train samples=748 batches=46
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] condition=acd1k split=val samples=230 batches=15

[Model] Loading SegFormer-B2 (ADE20K pretrained)...
Loading weights: 100% 380/380 [00:00<00:00, 851.83it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]
SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key         

In [44]:
import json

with open('outputs/exp2/sweep_s2_lr1e5/history.json') as f:
    history = json.load(f)

best = max(history, key=lambda x: x['val_mIoU'])
print(f"Stage 2 sweep (lr=1e-5):")
print(f"  Best epoch : {best['epoch']}")
print(f"  val mIoU   : {best['val_mIoU']:.4f}")
print(f"  val F1     : {best['val_F1']:.4f}")
print(f"  val MAE    : {best['val_MAE']:.4f}")

Stage 2 sweep (lr=1e-5):
  Best epoch : 16
  val mIoU   : 0.8602
  val F1     : 0.8527
  val MAE    : 0.0484


In [ ]:
# Stage 2 sweep run 3 (lr=5e-6)
!PYTHONPATH=/content/AI-final-project/src \
 python src/train_exp2.py \
    --stage 2 \
    --lr 5e-6 --epochs 20 --batch_size 16 --accum_steps 1 \
    --stage1_weights outputs/exp2/stage1/best_model.pth \
    --data_root data/ --splits_dir splits/ \
    --output_dir outputs/exp2/sweep_s2_lr5e6

Device: cuda
Config saved → outputs/exp2/sweep_s2_lr5e6/config.json
{
  "stage": 2,
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp2/sweep_s2_lr5e6",
  "encoder_lr": 0.0001,
  "head_lr": 0.001,
  "lr": 5e-06,
  "stage1_weights": "outputs/exp2/stage1/best_model.pth",
  "weight_decay": 0.0001,
  "epochs": 20,
  "batch_size": 16,
  "accum_steps": 1,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [ACD1K] 748 images (folder-scan mode)
  [DataLoader] condition=acd1k split=train samples=748 batches=46
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] condition=acd1k split=val samples=230 batches=15

[Model] Loading SegFormer-B2 (ADE20K pretrained)...
Loading weights: 100% 380/380 [00:00<00:00, 806.34it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]
SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key         

In [46]:
import json

with open('outputs/exp2/sweep_s2_lr5e6/history.json') as f:
    history = json.load(f)

best = max(history, key=lambda x: x['val_mIoU'])
print(f"Stage 2 sweep (lr=5e-6):")
print(f"  Best epoch : {best['epoch']}")
print(f"  val mIoU   : {best['val_mIoU']:.4f}")
print(f"  val F1     : {best['val_F1']:.4f}")
print(f"  val MAE    : {best['val_MAE']:.4f}")

Stage 2 sweep (lr=5e-6):
  Best epoch : 20
  val mIoU   : 0.8548
  val F1     : 0.8458
  val MAE    : 0.0509


---
## 8. Sweep Results — Pick Best LR

In [ ]:
# Compare all Stage 2 sweep results
import json, os

runs = {
    'sweep_s2_lr1e4': ('outputs/exp2/sweep_s2_lr1e4/history.json', 1e-4),
    'sweep_s2_lr1e5': ('outputs/exp2/sweep_s2_lr1e5/history.json', 1e-5),
    'sweep_s2_lr5e6': ('outputs/exp2/sweep_s2_lr5e6/history.json', 5e-6),
}

best_overall = None
best_lr = None

print("=== Experiment 2 — Stage 2 Sweep Results ===")

for name, (path, lr) in runs.items():
    if os.path.exists(path):
        with open(path) as f:
            h = json.load(f)

        best = max(h, key=lambda x: x['val_mIoU'])

        print(f"{name:<20} mIoU={best['val_mIoU']:.4f}  "
              f"F1={best['val_F1']:.4f}  "
              f"MAE={best['val_MAE']:.4f}  "
              f"@ ep{best['epoch']}")

        if best_overall is None or best['val_mIoU'] > best_overall['val_mIoU']:
            best_overall = best
            best_lr = lr

    else:
        print(f"{name:<20} NOT FOUND")

print(f"Best LR from sweep: {best_lr:.0e}")

=== Experiment 2 — Stage 2 Sweep Results ===
sweep_s2_lr1e4       mIoU=0.8682  F1=0.8622  MAE=0.0427  @ ep12
sweep_s2_lr1e5       mIoU=0.8602  F1=0.8527  MAE=0.0484  @ ep16
sweep_s2_lr5e6       mIoU=0.8548  F1=0.8458  MAE=0.0509  @ ep20
Best LR from sweep: 1e-04


---
## 9. Stage 2 — Final Training Run

Running the full 50-epoch training with the best learning rate from the above sweep.
Adjust the `--lr` value below based on your sweep results.

In [ ]:
# Stage 2 final training (50 epochs, with best LR from above sweep)
!PYTHONPATH=/content/AI-final-project/src python src/train_exp2.py \
    --stage 2 \
    --lr {best_lr} \
    --epochs 50 \
    --batch_size 16 \
    --accum_steps 1 \
    --stage1_weights outputs/exp2/stage1/best_model.pth \
    --data_root data/ \
    --splits_dir splits/ \
    --output_dir outputs/exp2/final

Device: cuda
Config saved → outputs/exp2/final/config.json
{
  "stage": 2,
  "data_root": "data/",
  "splits_dir": "splits/",
  "output_dir": "outputs/exp2/final",
  "encoder_lr": 0.0001,
  "head_lr": 0.001,
  "lr": 0.0001,
  "stage1_weights": "outputs/exp2/stage1/best_model.pth",
  "weight_decay": 0.0001,
  "epochs": 50,
  "batch_size": 16,
  "accum_steps": 1,
  "patience": 10,
  "num_workers": 2,
  "seed": 42
}

[DataLoaders]
  [ACD1K] 748 images (folder-scan mode)
  [DataLoader] condition=acd1k split=train samples=748 batches=46
  [Splits] ACD1K val: 230 images (from acd1k_splits.json)
  [ACD1K] 230 images (split-file mode)
  [DataLoader] condition=acd1k split=val samples=230 batches=15

[Model] Loading SegFormer-B2 (ADE20K pretrained)...
Loading weights: 100% 380/380 [00:00<00:00, 933.89it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]
SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key                          

---
## 10. Results & Export

In [ ]:
# Final results
import json

with open('outputs/exp2/final/history.json') as f:
    h = json.load(f)
best = max(h, key=lambda x: x['val_mIoU'])
print(f"Experiment 2 — Final Stage 2 Results:")
print(f"  Best epoch : {best['epoch']}")
print(f"  val mIoU   : {best['val_mIoU']:.4f}")
print(f"  val F1     : {best['val_F1']:.4f}")
print(f"  val MAE    : {best['val_MAE']:.4f}")
print(f"  val loss   : {best['val_loss']:.4f}")

Experiment 2 — Final Stage 2 Results:
  Best epoch : 9
  val mIoU   : 0.8698
  val F1     : 0.8638
  val MAE    : 0.0423
  val loss   : 0.2585


In [ ]:
# Full Experiment 2 results summary
import json, os

runs = {
    'stage1':          'outputs/exp2/stage1/history.json',
    'sweep_s2_lr1e4':  'outputs/exp2/sweep_s2_lr1e4/history.json',
    'sweep_s2_lr1e5':  'outputs/exp2/sweep_s2_lr1e5/history.json',
    'sweep_s2_lr5e6':  'outputs/exp2/sweep_s2_lr5e6/history.json',
    'final':           'outputs/exp2/final/history.json',
}

print("=== Experiment 2 Full Results ===")
for name, path in runs.items():
    if os.path.exists(path):
        with open(path) as f:
            h = json.load(f)
        best = max(h, key=lambda x: x['val_mIoU'])
        print(f"{name:<20} mIoU={best['val_mIoU']:.4f}  "
              f"F1={best['val_F1']:.4f}  "
              f"MAE={best['val_MAE']:.4f}  "
              f"@ ep{best['epoch']}")
    else:
        print(f"{name:<20} NOT FOUND")

=== Experiment 2 Full Results ===
stage1               mIoU=0.8944  F1=0.8464  MAE=0.0162  @ ep43
sweep_s2_lr1e4       mIoU=0.8682  F1=0.8622  MAE=0.0427  @ ep12
sweep_s2_lr1e5       mIoU=0.8602  F1=0.8527  MAE=0.0484  @ ep16
sweep_s2_lr5e6       mIoU=0.8548  F1=0.8458  MAE=0.0509  @ ep20
final                mIoU=0.8698  F1=0.8638  MAE=0.0423  @ ep9


In [ ]:
# Download final checkpoint
from google.colab import files
files.download('outputs/exp2/final/best_model.pth')
files.download('outputs/exp2/final/history.json')

[main d1ed72a] Exp2: Stage1 pretrain + Stage2 sweep + final (transfer learning)
 12 files changed, 1988 insertions(+)
 create mode 160000 AI-final-project
 create mode 100644 outputs/exp2/final/config.json
 create mode 100644 outputs/exp2/final/history.json
 create mode 100644 outputs/exp2/stage1/config.json
 create mode 100644 outputs/exp2/stage1/history.json
 create mode 100644 outputs/exp2/sweep_s2_lr1e4/config.json
 create mode 100644 outputs/exp2/sweep_s2_lr1e4/history.json
 create mode 100644 outputs/exp2/sweep_s2_lr1e5/config.json
 create mode 100644 outputs/exp2/sweep_s2_lr1e5/history.json
 create mode 100644 outputs/exp2/sweep_s2_lr5e6/config.json
 create mode 100644 outputs/exp2/sweep_s2_lr5e6/history.json
 create mode 100644 src/train_exp2.py
fatal: could not read Username for 'https://github.com': No such device or address


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>